<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/experiments/first_passage/duffing_oscillator/duffing_mean_escape_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duffing oscillator mean escape time

This notebook compares five methods for the Duffing mean exit time. Read the problem in Section 1, then edit the labeled problem, model, sampling, and training cells in Sections 2–5. Run top to bottom: Section 6 creates a fresh run, saves its seeds and numerical data, and trains the selected methods; Section 7 evaluates the saved checkpoints and creates the complete run archive. Drive is optional, and the notebook passes its output folder between cells.


## 1. Problem

The stochastic Duffing oscillator is

$$
dX_1=X_2\,dt+\sqrt{2\epsilon}\,dB_1,
\qquad
dX_2=(-X_1-X_1^3-2\zeta X_2)\,dt+\sqrt{2\zeta}\,dB_2.
$$

For the first exit from the disk $\Omega=B_R(0)$, the mean escape time $\tau(x)=\mathbb E_x[T]$ solves

$$\mathcal L\tau=-1\quad\text{in }\Omega,\qquad \tau=0\quad\text{on }\partial\Omega,$$

$$\mathcal L\tau=x_2\tau_{x_1}-(x_1+x_1^3+2\zeta x_2)\tau_{x_2}
+\epsilon\tau_{x_1x_1}+\zeta\tau_{x_2x_2}.$$

No pretrained Duffing PINN result is bundled. Numerical values appear only after this notebook has been run.


The notebook trains a boundary-adapted neural approximation using both the PDE residual and Monte Carlo exit-time labels.

### Five methods compared

All five methods use the same Duffing equation, Monte Carlo labels, collocation points, network, and optimizer. They differ in their trial function and boundary treatment.

| Method | Trial function | Boundary treatment |
|---|---|---|
| Standard | $\tau_\theta=N_\theta$ | Penalizes sampled boundary values. |
| Boundary-adapted | $\tau_\theta=dN_\theta$ | Enforces zero boundary values in the trial function. |
| Over-enforced | $\tau_\theta=d^{3/2}N_\theta$ | Enforces zero boundary values with a higher power of the distance. |
| Under-enforced | $\tau_\theta=d^{1/2}N_\theta$ | Enforces zero boundary values with a lower power of the distance. |
| Variational/Nitsche | $\tau_\theta=N_\theta$ | Uses the weak PDE residual and Nitsche boundary terms. |

All five also use the same Monte Carlo solution-value loss. Section 5 defines the active loss terms.

## 2. Choose the run and set problem parameters

Edit the run controls, physical parameters, total training points, Monte Carlo reference, optimizer, and evaluation settings in their labeled cells.

### Runtime setup

In [ ]:
import json
import math
import random
import secrets
import time
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

# Runtime settings are separate from the experiment parameters below.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE_NAME = "float64"  # Change to "float32" if needed.
if DTYPE_NAME not in ("float32", "float64"):
    raise ValueError("DTYPE_NAME must be 'float32' or 'float64'.")
torch.set_default_dtype(torch.float64 if DTYPE_NAME == "float64" else torch.float32)
print("Training device:", DEVICE)

### Run mode and output location

Choose a fresh run, resume an interrupted run, or evaluate saved checkpoints. Drive saving is optional in Colab.

In [ ]:
# Run and output
RUN_MODE = "new"                 # "new" trains a fresh run; "resume" or "evaluate" selects a saved run.
RUN_LABEL = "duffing"
SAVE_TO_DRIVE = False            # True mounts Google Drive when the run folder is prepared; False uses ./runs/duffing.
METHODS = ["standard", "adapted", "over", "under", "variational"]
RUN_LBFGS = False
DOWNLOAD_RUN_ARCHIVE = True  # Download the ZIP automatically when using local Colab storage.

### Duffing process and absorbing disk

The state is $X_t=(x_t,v_t)$. These values define the drift, diffusion, and disk in the exit-time PDE above.

In [ ]:
radius = 2.0
zeta = 0.08
epsilon = 1e-5
linear_stiffness = 1.0
cubic_stiffness = 1.0


## 3. Define the network and PDE

Edit `BaseNetwork` in the next code cell using ordinary PyTorch layers and initialization. Keep one scalar output per point. `DuffingModel` applies the selected trial function and trainable loss scales; the following cells define the problem-specific drift and PDE residual.


In [ ]:
def smooth_distance(points):
    return (radius**2 - points.square().sum(dim=1)) / (2 * radius)


# Edit the layers and initialization directly using standard PyTorch code.
# Keep one scalar output per input point.
class BaseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )
        self.layers.apply(self.initialize)

    @staticmethod
    def initialize(layer):
        if isinstance(layer, nn.Linear):
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, points):
        return self.layers(points)


class DuffingModel(nn.Module):
    def __init__(self, method):
        super().__init__()
        if method not in METHODS:
            raise ValueError(f"Unknown method {method!r}; choose from {METHODS}.")
        self.method = method
        if method == "adapted":
            self.power = 1.0
        elif method == "over":
            self.power = 1.5
        elif method == "under":
            self.power = 0.5
        else:
            self.power = None
        self.network = BaseNetwork()

        if method == "standard":
            loss_names = ("pde", "data", "bc")
        elif method == "variational":
            loss_names = ("var", "data", "bc")
        else:
            loss_names = ("pde", "data")
        self.log_scales = nn.ParameterDict({
            name: nn.Parameter(torch.tensor(
                nitsche_bc_log_scale_init if method == "variational" and name == "bc"
                else adaptive_log_scale_init
            ))
            for name in loss_names
        })

    def forward(self, points):
        raw = self.network(points)
        if raw.ndim == 1:
            raw = raw.unsqueeze(1)
        if raw.shape != (points.shape[0], 1):
            raise ValueError("BaseNetwork must return one scalar per point: [n] or [n, 1].")
        if self.power is None:
            return raw
        return raw * smooth_distance(points).clamp_min(0).pow(self.power).unsqueeze(1)


### Duffing drift

Euler–Maruyama advances both components until the path exits the disk or reaches the time cap. This plot checks that the same equation and noise strengths used by the PDE are being simulated.


In [ ]:
def duffing_drift(points):
    x1, x2 = points[..., 0], points[..., 1]
    return torch.stack((x2, -linear_stiffness * x1 - cubic_stiffness * x1**3 - 2 * zeta * x2), dim=-1)


### PDE residual

The strong residual below is \(\mathcal L\tau_\theta+1\). Edit it together with the SDE coefficients and drift if you change the process.

In [ ]:
def residual(model, points):
    x = points.detach().clone().requires_grad_(True)
    tau = model(x).squeeze(1)
    grad = torch.autograd.grad(tau.sum(), x, create_graph=True)[0]
    tau_xx = torch.autograd.grad(grad[:, 0].sum(), x, create_graph=True)[0][:, 0]
    tau_yy = torch.autograd.grad(grad[:, 1].sum(), x, create_graph=True)[0][:, 1]
    operator = (
        x[:, 1] * grad[:, 0]
        - (linear_stiffness * x[:, 0] + cubic_stiffness * x[:, 0]**3 + 2 * zeta * x[:, 1]) * grad[:, 1]
        + epsilon * tau_xx + zeta * tau_yy
    )
    return operator + 1.0


def pde_loss(model, points):
    return residual(model, points).square().mean()


def monte_carlo_label_loss(model, points, values):
    prediction = model(points).squeeze(1)
    return (prediction - values).square().mean()


## 4. Prepare samples and reference data

This section defines the domain samplers and reference solution. Training and evaluation point counts are set in Section 5; Monte Carlo anchor and path settings follow those choices in Section 6.

### Disk sampler

For a uniform sample by area in the disk, draw an angle uniformly and a radius \(R\sqrt U\), with \(U\sim\mathrm{Uniform}(0,1)\). This sampler supplies the fixed PDE and evaluation sets.

In [ ]:
def sample_disk(count, *, device=DEVICE):
    """Sample uniformly by area from the absorbing disk."""
    angle = 2 * math.pi * torch.rand(count, device=device)
    radial = radius * torch.sqrt(torch.rand(count, device=device))
    return torch.stack((radial * torch.cos(angle), radial * torch.sin(angle)), dim=1)


def sample_boundary(count, *, device=DEVICE):
    """Sample uniformly in angle on the absorbing circle."""
    angle = 2 * math.pi * torch.rand(count, device=device)
    return torch.stack((radius * torch.cos(angle), radius * torch.sin(angle)), dim=1)

## 5. Define losses and training settings

Edit these values for this part of the experiment.

Point-count cells state the total number of points in each fixed set; the sets are sampled once and reused throughout training. The optimizer, learning-rate schedule, optional L-BFGS stage, and loss terms are editable in the code cells below. The editable per-method training loop follows the loss definitions below; Section 6 runs it after preparing or loading the shared data.


Every epoch uses the saved collocation points and the same Monte Carlo labels for each selected method. The following cells expose the optimizer, schedule, method controls, and loss calculations.

### Total points in the training, evaluation, and optional L-BFGS sets

These are fixed set sizes, not mini-batch sizes. Each Adam update uses the full saved collocation set; the standard and variational methods also use the saved boundary set. Evaluation and L-BFGS use separate sets.

In [ ]:
training_collocation_count = 512  # Total interior points in each training set.
training_boundary_count = 128       # Boundary points for standard and variational losses.
evaluation_interior_count = 512     # Fixed interior points for epoch-to-epoch reporting.
evaluation_boundary_count = 128     # Fixed boundary points for epoch-to-epoch reporting.
lbfgs_interior_count = 512          # Fixed interior points for the optional L-BFGS stage.
lbfgs_boundary_count = 128          # Fixed boundary points for the optional L-BFGS stage.

### Adam optimizer and learning-rate schedule

The exponential schedule reaches the configured final learning-rate multiplier at the last Adam epoch.

In [ ]:
learning_rate = 1e-2
adam_beta_1 = 0.9
adam_beta_2 = 0.999
adam_epsilon = 1e-8
weight_decay = 0.0
adam_epochs = 40
final_learning_rate_factor = 5e-2
report_every = 10
checkpoint_every = 10

### Adaptive weighting, Nitsche, and optional L-BFGS

These initial adaptive scales are trainable parameters. Nitsche settings apply to the variational method; L-BFGS runs only when enabled in Section 2.

In [ ]:
adaptive_log_scale_init = -0.5
nitsche_bc_log_scale_init = 3.0
n_test_functions = 50
test_length = radius / 5.0
lbfgs_iterations = 20
lbfgs_lr = 1.0
lbfgs_history_size = 100
lbfgs_tolerance_grad = 0.0
lbfgs_tolerance_change = 0.0

### Losses used by the five methods

In [ ]:
test_centers = None


def variational_residual(model, interior, boundary):
    interior = interior.detach().clone().requires_grad_(True)
    boundary = boundary.detach().clone().requires_grad_(True)
    u_int = model(interior).squeeze(1)
    grad_u = torch.autograd.grad(u_int.sum(), interior, create_graph=True)[0]

    delta = interior[:, None, :] - test_centers[None, :, :]
    phi = torch.exp(-delta.square().sum(dim=2) / (2 * test_length**2))
    grad_phi = -delta * phi[..., None] / test_length**2
    area = math.pi * radius**2
    drift = duffing_drift(interior.detach())
    diffusion = (
        epsilon * grad_u[:, None, 0] * grad_phi[:, :, 0]
        + zeta * grad_u[:, None, 1] * grad_phi[:, :, 1]
    )
    drift_term = -(drift * grad_u).sum(dim=1, keepdim=True) * phi
    weak = (diffusion + drift_term).mean(dim=0) * area - phi.mean(dim=0) * area

    delta_b = boundary[:, None, :] - test_centers[None, :, :]
    phi_b = torch.exp(-delta_b.square().sum(dim=2) / (2 * test_length**2))
    grad_phi_b = -delta_b * phi_b[..., None] / test_length**2
    normal = boundary / radius
    u_b = model(boundary).squeeze(1)
    grad_u_b = torch.autograd.grad(u_b.sum(), boundary, create_graph=True)[0]
    flux_u = epsilon * grad_u_b[:, 0] * normal[:, 0] + zeta * grad_u_b[:, 1] * normal[:, 1]
    flux_phi = (
        epsilon * grad_phi_b[:, :, 0] * normal[:, None, 0]
        + zeta * grad_phi_b[:, :, 1] * normal[:, None, 1]
    )
    perimeter = 2 * math.pi * radius
    consistency = -(flux_u[:, None] * phi_b).mean(dim=0) * perimeter
    symmetry = -(u_b[:, None] * flux_phi).mean(dim=0) * perimeter
    boundary_penalty = (u_b[:, None] * phi_b).mean(dim=0) * perimeter
    return weak + consistency + symmetry, boundary_penalty


def adaptive_objective(model, components):
    terms = [
        torch.exp(-2 * model.log_scales[name]) * value + model.log_scales[name]
        for name, value in components.items()
    ]
    return sum(terms)


def component_losses(model, method, interior, boundary, data_x, data_y):
    data = monte_carlo_label_loss(model, data_x, data_y)
    if method == "variational":
        weak, boundary_term = variational_residual(model, interior, boundary)
        return {"var": weak.square().mean(), "data": data, "bc": boundary_term.square().mean()}

    losses = {"pde": pde_loss(model, interior), "data": data}
    if method == "standard":
        losses["bc"] = model(boundary).squeeze(1).square().mean()
    return losses

### Editable per-method training loop

This function defines one method’s Adam loop and optional L-BFGS stage. Each Adam update uses the fixed training points and shared Monte Carlo labels; separate evaluation points select the best Adam checkpoint. Checkpoints and epoch metrics are saved as training proceeds. This cell only defines the function. Section 6 calls it after preparing or loading the data.

In [ ]:
def train_method(method):
    method_dir = run_dir / method
    if RUN_MODE == "new" and method_dir.exists():
        raise FileExistsError(f"Refusing to reuse method folder: {method_dir}")
    method_dir.mkdir(parents=True, exist_ok=(RUN_MODE == "resume"))
    model = DuffingModel(method).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        betas=(adam_beta_1, adam_beta_2),
        eps=adam_epsilon,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ExponentialLR(
        optimizer,
        gamma=final_learning_rate_factor ** (1.0 / adam_epochs),
    )
    history, start_epoch, best_score = [], 0, math.inf
    lbfgs_history = []
    last_path = method_dir / "last.pt"
    best_path = method_dir / "best_adam.pt"
    final_path = method_dir / "final.pt"
    checkpoint_settings = {**experiment_settings, "method": method}
    final_settings = {
        **checkpoint_settings, "lbfgs": RUN_LBFGS,
        "lbfgs_iterations": lbfgs_iterations,
    }

    if RUN_MODE == "resume" and final_path.exists():
        state = torch.load(final_path, map_location=DEVICE, weights_only=False)
        require_matching_settings(state, final_settings, final_path)
        model.load_state_dict(state["model"])
        restore_rng_state(state.get("rng_state"))
        print(f"{method}: this method is already complete; reusing its checkpoint.")
        return model, pd.DataFrame(state["history"])
    if RUN_MODE == "resume" and last_path.exists():
        state = torch.load(last_path, map_location=DEVICE, weights_only=False)
        require_matching_settings(state, checkpoint_settings, last_path)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        history, start_epoch, best_score = state["history"], state["epoch"], state["best_score"]
        restore_rng_state(state.get("rng_state"))
        print(f"{method}: resuming after epoch {start_epoch}")
    elif RUN_MODE == "resume" and best_path.exists():
        raise ValueError(f"{best_path} exists without a resumable last.pt; refusing an inexact resume.")

    started = time.time()
    for epoch in range(start_epoch + 1, adam_epochs + 1):
        model.train()
        optimizer.zero_grad()
        components = component_losses(
            model, method, training_interior, training_boundary, x_data, tau_data
        )
        objective = adaptive_objective(model, components)
        objective.backward()
        optimizer.step()
        scheduler.step()

        model.eval()
        validation = component_losses(
            model, method, evaluation_interior, evaluation_boundary, x_data, tau_data
        )
        score = float(adaptive_objective(model, validation).detach())
        row = {
            "epoch": epoch,
            "objective": float(objective.detach()),
            "checkpoint_objective": score,
            **{name: float(value.detach()) for name, value in components.items()},
            **{f"log_scale_{name}": float(value.detach()) for name, value in model.log_scales.items()},
        }
        history.append(row)
        if score < best_score:
            best_score = score
            atomic_torch_save({
                "model": model.state_dict(), "epoch": epoch, "score": score,
                "settings": checkpoint_settings,
            }, best_path)

        if epoch % report_every == 0 or epoch == 1 or epoch == adam_epochs:
            pieces = " | ".join(f"{name} {value:.3e}" for name, value in row.items() if name != "epoch")
            print(
                f"{method:11s} epoch {epoch:5d}/{adam_epochs} | {pieces} | "
                f"{time.time() - started:.1f}s",
                flush=True,
            )
        if epoch % checkpoint_every == 0 or epoch == 1 or epoch == adam_epochs:
            atomic_torch_save({
                "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "history": history,
                "epoch": epoch, "best_score": best_score,
                "settings": checkpoint_settings, "rng_state": capture_rng_state(),
            }, last_path)
            atomic_csv_save(pd.DataFrame(history), method_dir / "training_history.csv")
            write_run_manifest("training")

    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=False)["model"])
    lbfgs_history = run_lbfgs_stage(model, method, method_dir, x_data, tau_data, lbfgs_history)
    atomic_torch_save({
        "model": model.state_dict(), "history": history, "lbfgs_history": lbfgs_history,
        "method": method, "run_label": RUN_LABEL, "settings": final_settings,
        "rng_state": capture_rng_state(),
    }, final_path)
    history_frame = pd.DataFrame(history)
    atomic_csv_save(history_frame, method_dir / "training_history.csv")
    return model, history_frame

### Optional L-BFGS refinement

When `RUN_LBFGS` is enabled, this separate stage refines the best Adam checkpoint using the fixed L-BFGS points. It writes its call-by-call loss history. Leave it disabled to run Adam only.

In [ ]:
def run_lbfgs_stage(model, method, method_dir, x_data, tau_data, lbfgs_history):
    if RUN_LBFGS:
        optimizer_lbfgs = torch.optim.LBFGS(
            model.parameters(),
            lr=lbfgs_lr,
            max_iter=lbfgs_iterations,
            history_size=lbfgs_history_size,
            tolerance_grad=lbfgs_tolerance_grad,
            tolerance_change=lbfgs_tolerance_change,
            line_search_fn="strong_wolfe",
        )
        calls = 0

        def closure():
            nonlocal calls
            calls += 1
            optimizer_lbfgs.zero_grad()
            components = component_losses(
                model, method, lbfgs_interior, lbfgs_boundary, x_data, tau_data
            )
            objective = adaptive_objective(model, components)
            objective.backward()
            lbfgs_history.append({
                "call": calls, "objective": float(objective.detach()),
                **{name: float(value.detach()) for name, value in components.items()},
                **{f"log_scale_{name}": float(value.detach()) for name, value in model.log_scales.items()},
            })
            if calls % 100 == 0:
                atomic_csv_save(pd.DataFrame(lbfgs_history), method_dir / "lbfgs_history.csv")
                write_run_manifest("training")
                print(f"{method}: L-BFGS call {calls} objective {float(objective.detach()):.3e}")
            return objective

        optimizer_lbfgs.step(closure)
        atomic_csv_save(pd.DataFrame(lbfgs_history), method_dir / "lbfgs_history.csv")
    return lbfgs_history

## 6. Save data and train

For a new run, prepare this section’s shared points and Monte Carlo labels first; each file is saved in the run folder. The training cell then runs the selected methods. Resume and evaluation reuse saved arrays and checkpoints.

### Monte Carlo solution labels

The collocation points in Section 5 are used in the empirical PDE loss. These **anchors are a separate set of input locations** where the notebook estimates the solution by simulation. For mean escape time, each label is the average exit time from independent Euler–Maruyama paths started at that anchor. This is a noisy Feynman–Kac estimate; its squared difference from the network prediction is added as a solution-value data loss alongside the PDE residual. [Tepakbong et al. (2026)](https://arxiv.org/abs/2607.19167) studies this supervision.

Set the number of anchors as a fraction of the total collocation count. For example, a fraction of 0.05 with 512 collocation points gives about 26 anchors. Increasing the paths per anchor reduces Monte Carlo noise but takes longer. The time step controls each Euler–Maruyama update; the time cap limits how long a path runs. Paths still inside at the cap are recorded as censored. All five methods share the same saved anchors and labels.

### Monte Carlo settings

Set the anchor fraction, number of independent paths per anchor, time step, and simulation time cap below. The displayed anchor count is computed from the total collocation count set in Section 5.

In [ ]:
mc_anchor_fraction = 20 / training_collocation_count
mc_anchors = max(1, round(mc_anchor_fraction * training_collocation_count))
mc_paths = 32
mc_dt = 2e-3
mc_max_time = 5.0

In [ ]:
if RUN_MODE not in {"new", "resume", "evaluate"}:
    raise ValueError("RUN_MODE must be 'new', 'resume', or 'evaluate'.")

if SAVE_TO_DRIVE:
    if drive is None:
        raise RuntimeError("Google Drive saving is available in Colab; set SAVE_TO_DRIVE = False for local runs.")
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive", force_remount=False)
    output_base = Path("/content/drive/MyDrive/boundary-adapted-pinns-runs/duffing")
else:
    output_base = Path("runs/duffing")
output_base.mkdir(parents=True, exist_ok=True)

if RUN_MODE == "new":
    # These seeds are generated once for this new run and saved in its manifest.
    SEED = secrets.randbits(32)
    MC_SEED = secrets.randbits(32)
    run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
    run_dir = output_base / f"{RUN_LABEL}_{run_stamp}_seed{SEED}"
    run_dir.mkdir(parents=False, exist_ok=False)
    saved_manifest = None
else:
    candidates = sorted(
        (path for path in output_base.glob(f"{RUN_LABEL}_*") if path.is_dir() and (path / "run_manifest.json").is_file()),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No saved run for {RUN_LABEL!r} was found in {output_base}.")
    if len(candidates) == 1:
        run_dir = candidates[0]
    else:
        print("Choose the saved run:")
        for number, path in enumerate(candidates, start=1):
            print(f"  {number}. {path.name}")
        choice = int(input(f"Run number [1-{len(candidates)}]: "))
        if not 1 <= choice <= len(candidates):
            raise ValueError("Choose one of the listed run numbers.")
        run_dir = candidates[choice - 1]
    manifest_path = run_dir / "run_manifest.json"
    saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    SEED = int(saved_manifest["seed"])
    MC_SEED = int(saved_manifest.get("mc_seed", saved_manifest.get("monte_carlo_seed")))

RESUME = RUN_MODE == "resume"
LOAD_SAVED_MODELS = RUN_MODE == "evaluate"
print(f"Mode: {RUN_MODE} | output: {run_dir} | training seed: {SEED} | Monte Carlo seed: {MC_SEED}")

def seed_all(seed):
    random.seed(int(seed))
    np.random.seed(int(seed) % (2**32))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

def atomic_torch_save(payload, path):
    temporary_path = path.with_name(path.name + ".tmp")
    torch.save(payload, temporary_path)
    temporary_path.replace(path)


def atomic_csv_save(frame, path):
    temporary_path = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def capture_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }

def restore_rng_state(state):
    if not state:
        raise ValueError("Checkpoint has no random-generator state; exact resume is unavailable.")
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all([item.cpu() for item in state["cuda"]])

seed_all(SEED)


### Run record

The next cell writes `run_manifest.json` in the selected run folder. It records the run seeds, the values that affect the experiment, its status, and the names of saved files. For resume or evaluation, it checks that the selected folder was created with matching settings and stops if they differ. Change experiment values in the cells where those values are introduced; this cell only records and checks them.

In [ ]:
experiment_settings = {
    "workflow_version": 3,
    "objective": "method-specific PDE or variational loss plus Monte Carlo label loss",
    "run_label": RUN_LABEL, "seed": SEED, "mc_seed": MC_SEED,
    "methods": list(METHODS),
    "radius": radius, "zeta": zeta, "epsilon": epsilon,
    "linear_stiffness": linear_stiffness, "cubic_stiffness": cubic_stiffness,
    "training_collocation_count": training_collocation_count,
    "training_boundary_count": training_boundary_count,
    "evaluation_interior_count": evaluation_interior_count,
    "evaluation_boundary_count": evaluation_boundary_count,
    "lbfgs_interior_count": lbfgs_interior_count,
    "lbfgs_boundary_count": lbfgs_boundary_count,
    "mc_anchors": mc_anchors, "mc_anchor_fraction": mc_anchor_fraction, "mc_paths": mc_paths,
    "mc_dt": mc_dt, "mc_max_time": mc_max_time,
    "adaptive_log_scale_init": adaptive_log_scale_init,
    "nitsche_bc_log_scale_init": nitsche_bc_log_scale_init,
    "n_test_functions": n_test_functions, "test_length": test_length,
    "network_layers": [2, 64, 64, 1],
    "network_activation": "tanh",
    "network_initialization": "xavier_normal",
    "learning_rate": learning_rate, "adam_beta_1": adam_beta_1, "adam_beta_2": adam_beta_2,
    "adam_epsilon": adam_epsilon, "weight_decay": weight_decay, "adam_epochs": adam_epochs,
    "final_learning_rate_factor": final_learning_rate_factor,
    "run_lbfgs": RUN_LBFGS, "lbfgs_iterations": lbfgs_iterations, "lbfgs_lr": lbfgs_lr,
    "lbfgs_history_size": lbfgs_history_size,
    "lbfgs_tolerance_grad": lbfgs_tolerance_grad, "lbfgs_tolerance_change": lbfgs_tolerance_change,
    "device": str(DEVICE), "dtype": DTYPE_NAME,
}
experiment_settings = json.loads(json.dumps(experiment_settings))

def require_matching_settings(record, expected, saved_path):
    saved = record.get("settings", record)
    if any(saved.get(key) != value for key, value in expected.items()):
        raise ValueError(
            f"{saved_path} was created with different settings. "
            "Restore those settings or start a new run."
        )

manifest_path = run_dir / "run_manifest.json"
if RUN_MODE == "new":
    write_run_manifest_record = {
        "schema_version": 2,
        "run_id": run_dir.name,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "seed": SEED,
        "mc_seed": MC_SEED,
        "monte_carlo_seed": MC_SEED,
        "settings": experiment_settings,
        "status": "created",
        "artifacts": [],
    }
    with manifest_path.open("x", encoding="utf-8") as stream:
        json.dump(write_run_manifest_record, stream, indent=2)
else:
    write_run_manifest_record = json.loads(manifest_path.read_text(encoding="utf-8"))
    require_matching_settings(write_run_manifest_record, experiment_settings, manifest_path)

def write_run_manifest(status):
    write_run_manifest_record["status"] = status
    write_run_manifest_record["updated_utc"] = datetime.now(timezone.utc).isoformat()
    write_run_manifest_record["artifacts"] = sorted(
        str(path.relative_to(run_dir))
        for path in run_dir.rglob("*")
        if path.is_file() and path != manifest_path and ".tmp" not in path.name
    )
    temporary_path = manifest_path.with_name(manifest_path.name + ".tmp")
    temporary_path.write_text(json.dumps(write_run_manifest_record, indent=2), encoding="utf-8")
    temporary_path.replace(manifest_path)

def atomic_npz_save(path, **arrays):
    temporary_path = path.with_name(path.name + ".tmp")
    with temporary_path.open("wb") as stream:
        np.savez_compressed(stream, **arrays)
    temporary_path.replace(path)

def atomic_figure_save(figure, path, **kwargs):
    temporary_path = path.with_name(path.stem + ".tmp" + path.suffix)
    figure.savefig(temporary_path, **kwargs)
    temporary_path.replace(path)

def require_artifact_files(paths):
    missing = [path for path in paths if not path.is_file() or path.stat().st_size == 0]
    if missing:
        raise RuntimeError("Missing or empty run outputs: " + ", ".join(map(str, missing)))

write_run_manifest("preparing")
print(f"Run folder: {run_dir} | seed: {SEED} | Monte Carlo seed: {MC_SEED}")


### Prepare or load Monte Carlo labels

For a new run, the next cell simulates paths from the saved anchors and writes estimated exit times and censoring fractions. Resume and evaluation load those values; they do not simulate another reference set. The following cell prepares fixed training/evaluation points. Neither preparation cell starts neural-network training.

In [ ]:
@torch.no_grad()
def monte_carlo_exit_time(starts, n_paths, dt, max_time):
    starts = starts.to(DEVICE)
    count = starts.shape[0]
    positions = starts[:, None, :].expand(count, n_paths, 2).clone()
    elapsed = torch.zeros(count, n_paths, device=DEVICE)
    alive = torch.ones(count, n_paths, dtype=torch.bool, device=DEVICE)
    steps = int(max_time / dt)
    interval = max(1, steps // 100)
    noise_scale = torch.tensor([math.sqrt(2 * epsilon * dt), math.sqrt(2 * zeta * dt)], device=DEVICE)
    started = time.time()
    for step in range(steps):
        if not alive.any():
            break
        current = positions[alive]
        positions[alive] = current + duffing_drift(current) * dt + torch.randn_like(current) * noise_scale
        elapsed[alive] += dt
        active_indices = alive.nonzero(as_tuple=False)
        exited = positions[alive].square().sum(dim=1) >= radius**2
        if exited.any():
            ids = active_indices[exited]
            alive[ids[:, 0], ids[:, 1]] = False
        if step % interval == 0 or step + 1 == steps:
            print(
                f"MC step {step + 1:7d}/{steps} | "
                f"alive {alive.float().mean():6.2%} | {time.time() - started:6.1f}s",
                flush=True,
            )
    return elapsed.mean(dim=1), alive.float().mean(dim=1)


write_run_manifest("preparing")
mc_path = run_dir / "monte_carlo_reference.pt"
mc_settings = {
    "format_version": 2,
    "seed": SEED, "mc_seed": MC_SEED,
        "monte_carlo_seed": MC_SEED, "radius": radius, "zeta": zeta, "epsilon": epsilon,
    "anchors": mc_anchors, "anchor_fraction": mc_anchor_fraction, "paths": mc_paths, "dt": mc_dt, "max_time": mc_max_time,
}
if RUN_MODE in {"resume", "evaluate"}:
    if not mc_path.is_file():
        raise FileNotFoundError(f"Saved Monte Carlo data is missing: {mc_path}")
    mc_record = torch.load(mc_path, map_location=DEVICE, weights_only=False)
    require_matching_settings(mc_record, mc_settings, mc_path)
    mc_x = mc_record["x"].to(DEVICE)
    mc_tau = mc_record["tau"].to(DEVICE)
    mc_censored = mc_record["censored"].to(DEVICE)
    print("Loaded Monte Carlo reference:", mc_path)
else:
    if mc_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing Monte Carlo data: {mc_path}")
    seed_all(MC_SEED)
    mc_x = sample_disk(mc_anchors)
    mc_tau, mc_censored = monte_carlo_exit_time(mc_x, mc_paths, mc_dt, mc_max_time)
    monte_carlo_record = {
        "x": mc_x.cpu(),
        "tau": mc_tau.cpu(),
        "censored": mc_censored.cpu(),
        "settings": mc_settings,
    }
    atomic_torch_save(monte_carlo_record, mc_path)
    print("Saved Monte Carlo reference:", mc_path)

mc_frame = pd.DataFrame({
    "x1": mc_x[:, 0].cpu().numpy(), "x2": mc_x[:, 1].cpu().numpy(),
    "mean exit time": mc_tau.cpu().numpy(),
    "censored fraction": mc_censored.cpu().numpy(),
})
atomic_csv_save(mc_frame, run_dir / "monte_carlo_reference.csv")
write_run_manifest("prepared")

display(mc_frame.describe())


### Prepare fixed training and evaluation points

In [ ]:
training_points_path = run_dir / "training_points.pt"

if RUN_MODE in {"resume", "evaluate"}:
    if not training_points_path.is_file():
        raise FileNotFoundError(f"Saved training points are missing: {training_points_path}")
    point_record = torch.load(training_points_path, map_location=DEVICE, weights_only=False)
    require_matching_settings(point_record, experiment_settings, training_points_path)
    saved_labels = point_record["label_points"].to(DEVICE)
    if not torch.equal(saved_labels, mc_x):
        raise ValueError("Saved training points and Monte Carlo reference contain different labels.")
    training_interior = point_record["training_interior"].to(DEVICE)
    training_boundary = point_record["training_boundary"].to(DEVICE)
    evaluation_interior = point_record["evaluation_interior"].to(DEVICE)
    evaluation_boundary = point_record["evaluation_boundary"].to(DEVICE)
    lbfgs_interior = point_record["lbfgs_interior"].to(DEVICE)
    lbfgs_boundary = point_record["lbfgs_boundary"].to(DEVICE)
    test_centers = point_record["test_centers"].to(DEVICE)
    print("Loaded this run's labels and point sets:", training_points_path)
else:
    if training_points_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing run data: {training_points_path}")
    seed_all(MC_SEED)
    training_interior = sample_disk(training_collocation_count)
    training_boundary = sample_boundary(training_boundary_count)
    evaluation_interior = sample_disk(evaluation_interior_count)
    evaluation_boundary = sample_boundary(evaluation_boundary_count)
    lbfgs_interior = sample_disk(lbfgs_interior_count)
    lbfgs_boundary = sample_boundary(lbfgs_boundary_count)
    test_centers = sample_disk(n_test_functions)
    point_record = {
        "collocation": training_interior.cpu(),
        "evaluation": evaluation_interior.cpu(),
        "label_points": mc_x.cpu(),
        "label_values": mc_tau.cpu(),
        "label_censored_fraction": mc_censored.cpu(),
        "training_interior": training_interior.cpu(),
        "training_boundary": training_boundary.cpu(),
        "evaluation_interior": evaluation_interior.cpu(),
        "evaluation_boundary": evaluation_boundary.cpu(),
        "lbfgs_interior": lbfgs_interior.cpu(),
        "lbfgs_boundary": lbfgs_boundary.cpu(),
        "test_centers": test_centers.cpu(),
        "settings": experiment_settings,
    }
    atomic_torch_save(point_record, training_points_path)
    print("Saved this run's labels and point sets:", training_points_path)

x_data, tau_data, data_censored = mc_x, mc_tau, mc_censored
atomic_npz_save(
    run_dir / "training_points.npz",
    collocation=training_interior.cpu().numpy(),
    evaluation=evaluation_interior.cpu().numpy(),
    label_points=x_data.cpu().numpy(),
    label_values=tau_data.cpu().numpy(),
    label_censored_fraction=data_censored.cpu().numpy(),
    training_interior=training_interior.cpu().numpy(),
    training_boundary=training_boundary.cpu().numpy(),
    evaluation_interior=evaluation_interior.cpu().numpy(),
    evaluation_boundary=evaluation_boundary.cpu().numpy(),
    lbfgs_interior=lbfgs_interior.cpu().numpy(),
    lbfgs_boundary=lbfgs_boundary.cpu().numpy(),
    test_centers=test_centers.cpu().numpy(),
    training_seed=np.asarray(SEED, dtype=np.uint64),
    monte_carlo_seed=np.asarray(MC_SEED, dtype=np.uint64),
)
seed_all(SEED)
write_run_manifest("prepared")

### Run or load the selected methods

Run the next cell after data preparation. In `new` or `resume` mode it calls the training loop for each selected method; in `evaluate` mode it loads saved checkpoints.

In [ ]:
trained, histories = {}, {}
if RUN_MODE in {"new", "resume"}:
    write_run_manifest("training")
    for method in METHODS:
        trained[method], histories[method] = train_method(method)
        print(f"{method}: training finished")
    write_run_manifest("trained")
elif RUN_MODE == "evaluate":
    write_run_manifest("evaluating")
    for method in METHODS:
        method_dir = run_dir / method
        final_path = method_dir / "final.pt"
        if not final_path.is_file():
            raise FileNotFoundError(f"Completed model is missing: {final_path}")
        state = torch.load(final_path, map_location=DEVICE, weights_only=False)
        expected = {
            **experiment_settings, "method": method, "lbfgs": RUN_LBFGS,
            "lbfgs_iterations": lbfgs_iterations,
        }
        require_matching_settings(state, expected, final_path)
        model = DuffingModel(method).to(DEVICE)
        model.load_state_dict(state["model"])
        trained[method] = model
        histories[method] = pd.DataFrame(state["history"])
        print(f"{method}: loaded completed model")
    for method, history in histories.items():
        atomic_csv_save(history, run_dir / method / "training_history.csv")

## 7. Evaluate and report

The notebook compares each method with the shared Monte Carlo labels at their starting points and reports the solution fields and anchor errors. Evaluation saves the numerical arrays, metrics, and figures alongside the checkpoints, histories, samples, and labels. It then creates a ZIP containing the complete run. With Drive selected, the ZIP stays beside the run; with local Colab storage, `DOWNLOAD_RUN_ARCHIVE` controls automatic download.

### Evaluation grid and figure resolution

These settings control the displayed solution and error plots. They do not change the saved training points, Monte Carlo labels, or fitted models. The evaluation cell records them in the run folder.

In [ ]:
evaluation_grid_n = 180
plot_dpi = 180

### Compare predictions with the saved labels

The five predictions are compared with the same Monte Carlo labels at their starting points. These are the labels used during training, so evaluation does not launch a second simulation.

### Learned solutions and anchor errors

The solution panels use one color scale across all five methods. The anchor comparison uses common axis limits, and the absolute-error panels share one vertical scale.

In [ ]:
if trained:
    write_run_manifest("evaluating")
    evaluation_settings = {
        "grid_size": evaluation_grid_n,
        "plot_dpi": plot_dpi,
    }
    (run_dir / "evaluation_settings.json").write_text(
        json.dumps(evaluation_settings, indent=2) + "\n", encoding="utf-8"
    )
    grid_n = evaluation_grid_n
    gx, gy = np.meshgrid(
        np.linspace(-radius, radius, grid_n),
        np.linspace(-radius, radius, grid_n),
    )
    disk_mask = gx**2 + gy**2 <= radius**2
    grid_points = np.column_stack((gx[disk_mask], gy[disk_mask]))
    grid_tensor = torch.as_tensor(grid_points, device=DEVICE)

    fields = {}
    anchor_predictions = {}
    for method, model in trained.items():
        model.eval()
        with torch.no_grad():
            prediction = model(grid_tensor).squeeze(1).cpu().numpy()
            anchor_prediction = model(x_data).squeeze(1).cpu().numpy()
        field = np.full_like(gx, np.nan)
        field[disk_mask] = prediction
        fields[method] = field
        anchor_predictions[method] = anchor_prediction

    reference = tau_data.cpu().numpy()
    censored_fraction = data_censored.cpu().numpy()
    comparison = pd.DataFrame({
        "x1": x_data[:, 0].cpu().numpy(),
        "x2": x_data[:, 1].cpu().numpy(),
        "Monte Carlo mean exit time": reference,
        "censored fraction": censored_fraction,
    })
    metrics = {}
    anchor_errors = {}
    for method, prediction in anchor_predictions.items():
        signed_error = prediction - reference
        absolute_error = np.abs(signed_error)
        anchor_errors[method] = absolute_error
        comparison[method] = prediction
        comparison[f"{method} absolute error"] = absolute_error
        metrics[method] = {
            "anchor_rmse": float(np.sqrt(np.mean(signed_error**2))),
            "anchor_mae": float(np.mean(absolute_error)),
            "anchor_relative_l2": float(
                np.linalg.norm(signed_error)
                / max(np.linalg.norm(reference), np.finfo(float).eps)
            ),
        }
    comparison.to_csv(run_dir / "anchor_comparison.csv", index=False)
    display(comparison)
    display(pd.DataFrame(metrics).T)

    arrays = {
        "gx": gx, "gy": gy, "disk_mask": disk_mask,
        "grid_points": grid_points,
        "monte_carlo_anchors": x_data.cpu().numpy(),
        "monte_carlo_exit_times": reference,
        "monte_carlo_censored_fraction": censored_fraction,
        "training_interior": training_interior.cpu().numpy(),
        "training_boundary": training_boundary.cpu().numpy(),
        "evaluation_interior": evaluation_interior.cpu().numpy(),
        "evaluation_boundary": evaluation_boundary.cpu().numpy(),
        "lbfgs_interior": lbfgs_interior.cpu().numpy(),
        "lbfgs_boundary": lbfgs_boundary.cpu().numpy(),
        "test_centers": test_centers.cpu().numpy(),
        "training_seed": np.asarray(SEED, dtype=np.uint64),
        "monte_carlo_seed": np.asarray(MC_SEED, dtype=np.uint64),
    }
    for method, field in fields.items():
        arrays[f"prediction_{method}"] = field
        arrays[f"anchor_prediction_{method}"] = anchor_predictions[method]
        arrays[f"absolute_anchor_error_{method}"] = anchor_errors[method]
        arrays[f"signed_anchor_error_{method}"] = anchor_predictions[method] - reference
        for column in histories[method].select_dtypes(include=[np.number]).columns:
            arrays[f"training_{method}_{column}"] = histories[method][column].to_numpy()
    results_path = run_dir / "numerical_results.npz"
    atomic_npz_save(results_path, **arrays)

    solution_min = min(float(values[disk_mask].min()) for values in fields.values())
    solution_max = max(float(values[disk_mask].max()) for values in fields.values())
    if solution_min == solution_max:
        solution_max = solution_min + np.finfo(float).eps
    solution_levels = np.linspace(solution_min, solution_max, 51)
    solution_figure, solution_axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    for axis, (method, values) in zip(solution_axes.flat, fields.items()):
        image = axis.contourf(
            gx, gy, np.ma.masked_where(~disk_mask, values),
            levels=solution_levels, vmin=solution_min, vmax=solution_max, extend="both",
        )
        axis.set_title(method)
        axis.set_aspect("equal")
        solution_figure.colorbar(image, ax=axis)
    for axis in solution_axes.flat[len(fields):]:
        axis.axis("off")
    solution_figure.suptitle("Duffing mean escape time: five solution fields")
    atomic_figure_save(solution_figure, run_dir / "solution_fields.png", dpi=plot_dpi)

    common_low = min(float(reference.min()), *(float(v.min()) for v in anchor_predictions.values()))
    common_high = max(float(reference.max()), *(float(v.max()) for v in anchor_predictions.values()))
    if common_low == common_high:
        common_high = common_low + np.finfo(float).eps
    anchor_figure, anchor_axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    for axis, (method, prediction) in zip(anchor_axes.flat, anchor_predictions.items()):
        axis.scatter(reference, prediction, c=censored_fraction, cmap="magma", vmin=0, vmax=1)
        axis.plot([common_low, common_high], [common_low, common_high], "k--")
        axis.set_xlim(common_low, common_high)
        axis.set_ylim(common_low, common_high)
        axis.set_aspect("equal", adjustable="box")
        axis.set_title(method)
        axis.set_xlabel("Monte Carlo mean exit time")
        axis.set_ylabel("PINN prediction")
    for axis in anchor_axes.flat[len(anchor_predictions):]:
        axis.axis("off")
    anchor_figure.suptitle("Duffing: predictions at shared Monte Carlo anchors")
    atomic_figure_save(anchor_figure, run_dir / "anchor_comparison.png", dpi=plot_dpi)

    error_limit = max(float(values.max()) for values in anchor_errors.values())
    error_limit = max(error_limit, np.finfo(float).eps)
    error_figure, error_axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    anchor_index = np.arange(len(reference))
    for axis, (method, values) in zip(error_axes.flat, anchor_errors.items()):
        axis.plot(anchor_index, values, marker="o", linewidth=1)
        axis.set_ylim(0.0, error_limit)
        axis.set_title(f"{method}: absolute error")
        axis.set_xlabel("Monte Carlo anchor")
        axis.set_ylabel("Absolute error")
    for axis in error_axes.flat[len(anchor_errors):]:
        axis.axis("off")
    error_figure.suptitle(f"Duffing anchor errors: shared scale [0, {error_limit:.3g}]")
    atomic_figure_save(error_figure, run_dir / "anchor_absolute_errors.png", dpi=plot_dpi)

    metrics_record = {
        "run_id": run_dir.name,
        "training_seed": SEED,
        "monte_carlo_seed": MC_SEED,
        "settings": experiment_settings,
        "methods": list(trained),
        "metrics": metrics,
        "solution_color_limits": [solution_min, solution_max],
        "anchor_comparison_limits": [common_low, common_high],
        "absolute_anchor_error_limits": [0.0, error_limit],
        "arrays_file": results_path.name,
    }
    metrics_path = run_dir / "numerical_metrics.json"
    temporary_metrics_path = metrics_path.with_name(metrics_path.name + ".tmp")
    temporary_metrics_path.write_text(json.dumps(metrics_record, indent=2) + "\n", encoding="utf-8")
    temporary_metrics_path.replace(metrics_path)

    write_run_manifest("evaluating")
    required_paths = [
        manifest_path, training_points_path, run_dir / "training_points.npz",
        run_dir / "monte_carlo_reference.pt", run_dir / "monte_carlo_reference.csv",
        results_path, metrics_path, run_dir / "anchor_comparison.csv",
        run_dir / "solution_fields.png", run_dir / "anchor_comparison.png",
        run_dir / "anchor_absolute_errors.png",
    ]
    for method in METHODS:
        method_dir = run_dir / method
        required_paths.extend([
            method_dir / "best_adam.pt", method_dir / "last.pt", method_dir / "final.pt",
            method_dir / "training_history.csv",
        ])
        if RUN_LBFGS:
            required_paths.append(method_dir / "lbfgs_history.csv")
    require_artifact_files(required_paths)

    with np.load(run_dir / "training_points.npz", allow_pickle=False) as saved_points:
        required_point_arrays = {
            "label_points", "label_values", "label_censored_fraction",
            "training_interior", "training_boundary",
            "evaluation_interior", "evaluation_boundary",
            "lbfgs_interior", "lbfgs_boundary", "test_centers",
            "training_seed", "monte_carlo_seed",
        }
        missing_points = required_point_arrays.difference(saved_points.files)
    if missing_points:
        raise RuntimeError("Training-point archive is incomplete: " + ", ".join(sorted(missing_points)))
    with np.load(results_path, allow_pickle=False) as saved_results:
        missing_results = {
            "grid_points", "monte_carlo_anchors", "monte_carlo_exit_times",
            "training_seed", "monte_carlo_seed",
        }.difference(saved_results.files)
        for method in METHODS:
            missing_results.update({
                f"prediction_{method}", f"anchor_prediction_{method}",
                f"absolute_anchor_error_{method}", f"training_{method}_epoch",
            }.difference(saved_results.files))
    if missing_results:
        raise RuntimeError("Numerical result archive is incomplete: " + ", ".join(sorted(missing_results)))

    write_run_manifest("complete")
    plt.show()

import shutil
archive_base = run_dir.parent / f"{run_dir.name}_archive"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=run_dir)
print("Complete run archive:", archive_path)
if not SAVE_TO_DRIVE and DOWNLOAD_RUN_ARCHIVE:
    try:
        from google.colab import files
    except ImportError:
        print("Automatic download is available in Colab; the ZIP is saved at the path above.")
    else:
        files.download(archive_path)


### Optional sample trajectory

The next settings and function produce one independent path for illustration. It does not change the saved Monte Carlo training labels.

In [ ]:
trajectory_start = (0.0, 0.0)
trajectory_dt = 1e-3
trajectory_max_time = 5.0
trajectory_seed_offset = 1

In [ ]:
def simulate_trajectory(
    start=trajectory_start, *, zeta_value=zeta, epsilon_value=epsilon,
    radius_value=radius, dt=trajectory_dt, max_time=trajectory_max_time, seed=SEED + trajectory_seed_offset,
):
    rng = np.random.default_rng(seed)
    point = np.asarray(start, dtype=float).copy()
    path = [point.copy()]
    exited = np.linalg.norm(point) >= radius_value
    noise = np.array([math.sqrt(2 * epsilon_value * dt), math.sqrt(2 * zeta_value * dt)])
    for _ in range(int(max_time / dt)):
        if exited:
            break
        drift_second = (
            -linear_stiffness * point[0]
            - cubic_stiffness * point[0]**3
            - 2 * zeta_value * point[1]
        )
        drift = np.array([point[1], drift_second])
        point = point + drift * dt + noise * rng.standard_normal(2)
        path.append(point.copy())
        exited = np.linalg.norm(point) >= radius_value
    return np.asarray(path), (len(path) - 1) * dt, exited


trajectory, trajectory_time, trajectory_exited = simulate_trajectory()
angle = np.linspace(0, 2 * np.pi, 300)
plt.figure(figsize=(6, 6))
plt.plot(trajectory[:, 0], trajectory[:, 1], lw=1)
plt.plot(radius * np.cos(angle), radius * np.sin(angle), "k--")
plt.scatter(*trajectory[0], label="start")
plt.scatter(*trajectory[-1], marker="x", label="last point")
plt.gca().set_aspect("equal")
plt.legend()
plt.title(f"Duffing trajectory: {'exit' if trajectory_exited else 'censored'} at t={trajectory_time:.3g}")
plt.show()


## 8. References
- J. Qian, L. Chen, and J. Q. Sun, [“A neural network solution of first-passage problems”](https://doi.org/10.1007/s10483-024-3189-8), *Applied Mathematics and Mechanics (English Edition)* 45(11), 2023–2036 (2024).
- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, [“Boundary-Adapted PINNs for Elliptic Dirichlet Problems”](https://arxiv.org/abs/2607.19167), 2026.
- N. Sukumar and A. Srivastava, [“Exact imposition of boundary conditions with distance functions”](https://doi.org/10.1016/j.cma.2021.114333), *CMAME* 389 (2022).
- M. Raissi, P. Perdikaris, and G. E. Karniadakis, [“Physics-informed neural networks”](https://doi.org/10.1016/j.jcp.2018.10.045), *JCP* 378 (2019).

- N. Tepakbong, H. Hu, C. Liu, and X. Zhou, “Taming the Loss Landscape of PINNs with Noisy Feynman–Kac Supervision: Operator Preconditioning and Non-Asymptotic Error Bounds,” ICML 2026. [arXiv preprint](https://arxiv.org/abs/2607.19167).